# ENOE variable validation companion

Issue #61. Inspect and rerun the source/staging and catalogue audit for the four reviewed quarters.
Run with the project Python 3.12 environment and the private local archives, database and manifest.
This notebook emits aggregates only; never commit execution outputs.

The exact checks live in `src/platform_work_reform/enoe_variable_audit.py` and are documented in
`docs/enoe-variable-validation.md`. This validates descriptive fields, not platform identification,
release-specific questionnaire routing, survey precision or IMSS comparability.


In [ ]:
import json
import os
from pathlib import Path

from platform_work_reform.config import Settings
from platform_work_reform.enoe_variable_audit import compare

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists())
os.chdir(root)
entries = json.loads(Path("artifacts/enoe-quarter-manifest.json").read_text())
report = compare(Settings.from_env(), entries)


In [ ]:
# Coverage, denominator and source agreement, without any person records.
[
    {
        "quarter": f"{q['year']}q{q['quarter']}",
        "source_sha256": q["source_sha256"],
        "source_staging_match": q["source_staging_match"],
        "counts": q["flow_counts"],
        "weighted_totals": q["flow_weighted_totals"],
    }
    for q in report["quarters"]
]


In [ ]:
# Non-substantive states remain separate; weighted counts are descriptive only.
[
    {
        "quarter": f"{q['year']}q{q['quarter']}",
        "field": field,
        "states": values["states"],
        "weighted_states": values["weighted_states"],
    }
    for q in report["quarters"]
    for field, values in q["fields"].items()
]


In [ ]:
# p4 routing evidence and medical-access consistency, without declaring reconstruction valid.
[
    {"quarter": f"{q['year']}q{q['quarter']}", "cross_tabs": q["cross_tabs"]}
    for q in report["quarters"]
]


In [ ]:
# Agreement with reviewed 2023 medical-access reconstruction rules; 2026 applicability remains qualified.
[
    {"quarter": f"{q['year']}q{q['quarter']}", "reference_checks": q["reference_rule_agreement"]}
    for q in report["quarters"]
]


Interpret blank responses using the routing discussion and limitations in the validation note.
The candidate-screen proposals in that note are for review; this notebook does not estimate them.
Retain original archive/catalogue hashes and use one explicit run per quarter. No source is modified.
